<a href="https://colab.research.google.com/github/JaymeManhica/AndroidFlutter/blob/master/transcricao_de_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# PIPELINE COMPLETO: LIMPEZA + TRANSCRIÇÃO AVANÇADA DE ÁUDIO
# Google Colab — Whisper + Diarização + Pós-processamento LLM
# ============================================================
# ANTES DE CORRER:
# 1. Activa GPU: Ambiente de execução -> Alterar tipo de ambiente
#    de execução -> GPU (T4 Gratuita) -> Guardar -> Reiniciar
# 2. Cria conta em https://huggingface.co e aceita termos em:
#    https://huggingface.co/pyannote/speaker-diarization-3.1
# 3. Gera token HF em: https://huggingface.co/settings/tokens
# ============================================================

# ════════════════════════════════════════════════════════════
# CONFIGURAÇÕES — edita apenas esta secção
# ════════════════════════════════════════════════════════════
MODELO_WHISPER    = "medium"   # tiny | base | small | medium | large
LINGUAGEM         = "pt"       # pt = Português
REMOVER_SILENCIO  = True       # True = limpar silêncios antes de transcrever
DIARIZACAO        = False       # True = identificar oradores (requer HF_TOKEN)
CORRIGIR_COM_LLM  = False      # True = corrigir com Claude API (requer ANTHROPIC_API_KEY)
HF_TOKEN          = ""         # Token HuggingFace — https://huggingface.co/settings/tokens
ANTHROPIC_API_KEY = ""         # API key Anthropic — https://console.anthropic.com
DURACAO_BLOCO_MIN = 15         # Minutos por bloco na remoção de silêncio
# ════════════════════════════════════════════════════════════

import subprocess, sys

# ---- Fixar versões incompatíveis antes de qualquer import ----
print("="*60)
print("PASSO 0: A preparar ambiente...")
print("="*60)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "pandas==2.2.2", "numpy<2.1", "numba==0.60.0"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "openai-whisper"], check=False)
subprocess.run(["apt-get", "install", "-y", "ffmpeg", "-q"],
    capture_output=True)
if DIARIZACAO:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "pyannote.audio"], check=False)
if CORRIGIR_COM_LLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "anthropic"], check=False)

import os, math, time
import torch
import whisper
from google.colab import files

# ---- Verificar GPU ----
print("\n" + "="*60)
print("VERIFICAÇÃO DE HARDWARE")
print("="*60)
if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
    USE_FP16 = True
else:
    print("AVISO: GPU não detectada — a correr em CPU (muito mais lento!)")
    print("Activa GPU em: Ambiente de execução -> Alterar tipo de ambiente -> GPU (T4)")
    USE_FP16 = False

# ---- Upload ----
print("\n" + "="*60)
print("UPLOAD DO FICHEIRO DE ÁUDIO")
print("="*60)
uploaded = files.upload()
nome_original = list(uploaded.keys())[0]
extensao = nome_original.split(".")[-1]
input_filename = "audio_entrada." + extensao
os.rename(nome_original, input_filename)
tamanho_mb = os.path.getsize(input_filename) / (1024*1024)
print(f"Ficheiro: {input_filename} ({tamanho_mb:.1f} MB)")


# ════════════════════════════════════════════════════════════
# ETAPA 1 — REMOÇÃO DE SILÊNCIOS (opcional)
# ════════════════════════════════════════════════════════════
audio_para_transcrever = input_filename

if REMOVER_SILENCIO:
    print("\n" + "="*60)
    print("ETAPA 1: A remover silêncios...")
    print("="*60)

    def obter_duracao(f):
        cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration",
               "-of", "default=noprint_wrappers=1:nokey=1", f]
        out = subprocess.run(cmd, capture_output=True, text=True)
        return float(out.stdout.strip())

    duracao_total = obter_duracao(input_filename)
    print(f"Duração total: {duracao_total/60:.1f} min ({duracao_total/3600:.2f}h)")

    bloco_seg = DURACAO_BLOCO_MIN * 60
    num_blocos = math.ceil(duracao_total / bloco_seg)
    print(f"A dividir em {num_blocos} blocos de ~{DURACAO_BLOCO_MIN} min cada\n")

    os.makedirs("blocos", exist_ok=True)
    filtro = ("silenceremove=stop_periods=-1:stop_duration=2.0:"
              "stop_threshold=-35dB:detection=peak")
    ficheiros_limpos = []

    for i in range(num_blocos):
        inicio = i * bloco_seg
        saida = f"blocos/parte_{i:03d}.mp3"
        print(f"  [{i+1}/{num_blocos}] min {inicio/60:.0f}–{(inicio+bloco_seg)/60:.0f}...", end=" ")

        cmd = ["ffmpeg", "-y", "-nostdin",
               "-ss", str(inicio), "-t", str(bloco_seg),
               "-i", input_filename,
               "-af", filtro,
               "-c:a", "libmp3lame", "-b:a", "128k", saida]
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            if r.returncode != 0:
                print(f"ERRO: {r.stderr[-200:]}")
                continue
            tam = os.path.getsize(saida) / (1024*1024)
            try:
                dur = obter_duracao(saida)
            except:
                dur = 0
            if tam < 0.01 or dur < 0.1:
                print(f"vazio (ignorado)")
            else:
                print(f"OK ({tam:.1f} MB, {dur:.0f}s)")
                ficheiros_limpos.append(saida)
        except subprocess.TimeoutExpired:
            print("TIMEOUT (ignorado)")

    if ficheiros_limpos:
        lista = "lista_blocos.txt"
        with open(lista, "w") as f:
            for fp in ficheiros_limpos:
                f.write(f"file '{os.path.abspath(fp)}'\n")
        audio_limpo = "audio_limpo.mp3"
        cmd_concat = ["ffmpeg", "-y", "-nostdin",
                      "-f", "concat", "-safe", "0",
                      "-i", lista, "-c", "copy", audio_limpo]
        r = subprocess.run(cmd_concat, capture_output=True, text=True, timeout=300)
        if r.returncode == 0:
            dur_final = obter_duracao(audio_limpo)
            print(f"\nAudio limpo: {dur_final/60:.1f} min "
                  f"(removidos {(duracao_total-dur_final)/60:.1f} min de silêncio)")
            audio_para_transcrever = audio_limpo
        else:
            print(f"Erro ao juntar blocos: {r.stderr[-300:]}")
            print("A usar áudio original...")
    else:
        print("Nenhum bloco válido. A usar áudio original...")


# ════════════════════════════════════════════════════════════
# ETAPA 2 — TRANSCRIÇÃO COM WHISPER
# ════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("ETAPA 2: A transcrever com Whisper...")
print("="*60)

INITIAL_PROMPT = """
Seminário da Electricidade de Moçambique, EDM, E.P., realizado em Maputo, Moçambique.
Contexto: Dia Internacional dos Arquivos.
Termos frequentes: gestão documental, arquivo, fundo documental, custódia,
descrição arquivística, conservação, preservação, digitalização,
Conselho Internacional de Arquivos, ICA, acesso à informação,
avaliação documental, tabela de selecção, instrumento de descrição,
plano de classificação, sistema de gestão electrónica de documentos, SGED,
Arquivo Histórico de Moçambique, Arquivo Nacional, palestrante, mesa redonda,
discurso de abertura, discurso de encerramento, boas-vindas.
Nomes moçambicanos: Jaime, Machava, Nhampule, Guambe, Sitoe, Cumbe,
Matsinhe, Mavie, Mondlane, Chissano, Nyusi, Manhiça, Maputo, Nampula,
Beira, Quelimane, Tete, Chimoio, Inhambane, Xai-Xai.
"""

print(f"A carregar modelo '{MODELO_WHISPER}'...")
modelo_whisper = whisper.load_model(MODELO_WHISPER)

inicio = time.time()
resultado = modelo_whisper.transcribe(
    audio_para_transcrever,
    language=LINGUAGEM,
    verbose=True,
    fp16=USE_FP16,
    initial_prompt=INITIAL_PROMPT,
    condition_on_previous_text=True,
    temperature=0.0,
    compression_ratio_threshold=2.4,
    no_speech_threshold=0.6,
)
duracao_proc = time.time() - inicio
print(f"\nTranscrição concluída em {duracao_proc/60:.1f} min!")

segmentos = resultado["segments"]
texto_bruto = resultado["text"].strip()

# Guardar texto corrido
with open("transcricao_bruta.txt", "w", encoding="utf-8") as f:
    f.write(texto_bruto)

# Guardar com timestamps
with open("transcricao_com_tempos.txt", "w", encoding="utf-8") as f:
    for seg in segmentos:
        s, e = seg["start"], seg["end"]
        hs,ms,ss = int(s//3600), int((s%3600)//60), int(s%60)
        he,me,se = int(e//3600), int((e%3600)//60), int(e%60)
        f.write(f"[{hs:02d}:{ms:02d}:{ss:02d} -> {he:02d}:{me:02d}:{se:02d}] "
                f"{seg['text'].strip()}\n")

print("Guardado: transcricao_bruta.txt | transcricao_com_tempos.txt")


# ════════════════════════════════════════════════════════════
# ETAPA 3 — DIARIZAÇÃO DE ORADORES
# ════════════════════════════════════════════════════════════
transcricao_final = texto_bruto

if DIARIZACAO:
    if not HF_TOKEN:
        print("\nAVISO: HF_TOKEN vazio — a saltar diarização.")
        print("Gera token em: https://huggingface.co/settings/tokens")
    else:
        print("\n" + "="*60)
        print("ETAPA 3: A identificar oradores...")
        print("="*60)
        try:
            from pyannote.audio import Pipeline

            pipeline = Pipeline.from_pretrained(
                "pyannote/speaker-diarization-3.1",
                use_auth_token=HF_TOKEN
            )
            if torch.cuda.is_available():
                pipeline = pipeline.to(torch.device("cuda"))

            print("A processar diarização...")
            diarizacao = pipeline(audio_para_transcrever)

            turnos = [{"inicio": t.start, "fim": t.end, "orador": o}
                      for t, _, o in diarizacao.itertracks(yield_label=True)]

            def orador_em(t):
                for turno in turnos:
                    if turno["inicio"] <= t <= turno["fim"]:
                        return turno["orador"]
                return "DESCONHECIDO"

            linhas = []
            orador_ant = None
            for seg in segmentos:
                t_meio = (seg["start"] + seg["end"]) / 2
                orador = orador_em(t_meio)
                s = seg["start"]
                hs,ms,ss = int(s//3600), int((s%3600)//60), int(s%60)
                ts = f"[{hs:02d}:{ms:02d}:{ss:02d}]"
                if orador != orador_ant:
                    linhas.append(f"\n{orador} {ts}")
                    orador_ant = orador
                linhas.append(f" {seg['text'].strip()}")

            transcricao_diarizada = "".join(linhas)
            with open("transcricao_com_oradores.txt", "w", encoding="utf-8") as f:
                f.write(transcricao_diarizada)
            transcricao_final = transcricao_diarizada
            print("Guardado: transcricao_com_oradores.txt")

        except Exception as ex:
            print(f"Erro na diarização: {ex}")
            print("A continuar sem diarização...")


# ════════════════════════════════════════════════════════════
# ETAPA 4 — CORRECÇÃO COM LLM (Claude)
# ════════════════════════════════════════════════════════════
if CORRIGIR_COM_LLM:
    if not ANTHROPIC_API_KEY:
        print("\nAVISO: ANTHROPIC_API_KEY vazia — a saltar correcção LLM.")
    else:
        print("\n" + "="*60)
        print("ETAPA 4: A corrigir com Claude...")
        print("="*60)
        import anthropic

        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

        def corrigir_bloco(texto, n, total):
            print(f"  Bloco {n}/{total}...", end=" ")
            r = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=4096,
                messages=[{"role": "user", "content": f"""És especialista em arquivística e gestão
documental em Moçambique. Corrige esta transcrição automática de um seminário da EDM
sobre o Dia Internacional dos Arquivos.

REGRAS:
- Corrige erros mantendo o sentido original
- Adiciona pontuação e parágrafos correctos
- Corrige nomes moçambicanos e termos técnicos de arquivo
- Mantém marcações de orador e timestamps se existirem
- Português de Moçambique (não Brasil)
- Não inventas conteúdo

TRANSCRIÇÃO:
{texto}

Devolve APENAS o texto corrigido."""}]
            )
            print("OK")
            return r.content[0].text

        palavras = transcricao_final.split()
        blocos = [" ".join(palavras[i:i+2000])
                  for i in range(0, len(palavras), 2000)]
        corrigidos = []
        for i, bloco in enumerate(blocos):
            try:
                corrigidos.append(corrigir_bloco(bloco, i+1, len(blocos)))
                time.sleep(1)
            except Exception as ex:
                print(f"Erro: {ex} — a manter original")
                corrigidos.append(bloco)

        transcricao_corrigida = "\n\n".join(corrigidos)
        with open("transcricao_corrigida.txt", "w", encoding="utf-8") as f:
            f.write(transcricao_corrigida)
        transcricao_final = transcricao_corrigida
        print("Guardado: transcricao_corrigida.txt")


# ════════════════════════════════════════════════════════════
# RESULTADO FINAL + DOWNLOAD
# ════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PIPELINE CONCLUÍDO!")
print("="*60)
print("\nFicheiros gerados:")
ficheiros_saida = [
    ("transcricao_bruta.txt",       "texto corrido, sem formatação"),
    ("transcricao_com_tempos.txt",  "com timestamps por segmento"),
    ("transcricao_com_oradores.txt","com identificação de oradores"),
    ("transcricao_corrigida.txt",   "corrigida pelo Claude LLM"),
    ("audio_limpo.mp3",             "áudio sem silêncios"),
]
for nome, desc in ficheiros_saida:
    if os.path.exists(nome):
        tam = os.path.getsize(nome) / (1024*1024)
        print(f"  {nome} ({tam:.1f} MB) — {desc}")

print("\nAMOSTRA DA TRANSCRIÇÃO (primeiros 800 caracteres):")
print("-"*60)
print(texto_bruto[:800])
print("...\n")

print("A iniciar downloads...")
for nome, _ in ficheiros_saida:
    if os.path.exists(nome):
        files.download(nome)

print("\nConcluído!")

# ============================================================
# REFERÊNCIA RÁPIDA — CONFIGURAÇÕES
# ============================================================
# MODELO_WHISPER:
#   tiny   (~1GB VRAM)  — muito rápido, menos preciso
#   base   (~1GB VRAM)  — rápido, razoável
#   small  (~2GB VRAM)  — bom equilíbrio (recomendado sem GPU)
#   medium (~5GB VRAM)  — melhor qualidade com GPU T4 ← padrão
#   large  (~10GB VRAM) — máxima precisão, requer GPU A100
#
# DIARIZACAO=True requer:
#   1. Conta em huggingface.co
#   2. Aceitar termos: huggingface.co/pyannote/speaker-diarization-3.1
#   3. Token em: huggingface.co/settings/tokens
#
# CORRIGIR_COM_LLM=True requer:
#   API key Anthropic: console.anthropic.com (tem custo por tokens)
#
# FINE-TUNING FUTURO:
#   Recolhe 30-60 min de áudio EDM com transcrição manual correcta
#   e usa HuggingFace Transformers para afinar o modelo Whisper:
#   github.com/huggingface/community-events/tree/main/whisper-fine-tuning-event
# ============================================================

PASSO 0: A preparar ambiente...

VERIFICAÇÃO DE HARDWARE
GPU detectada: Tesla T4

UPLOAD DO FICHEIRO DE ÁUDIO


Saving audio_sem_silencio.mp3 to audio_sem_silencio.mp3
Ficheiro: audio_entrada.mp3 (183.4 MB)

ETAPA 1: A remover silêncios...
Duração total: 200.3 min (3.34h)
A dividir em 14 blocos de ~15 min cada

  [1/14] min 0–15... OK (13.6 MB, 894s)
  [2/14] min 15–30... OK (13.7 MB, 900s)
  [3/14] min 30–45... OK (13.7 MB, 898s)
  [4/14] min 45–60... OK (13.7 MB, 900s)
  [5/14] min 60–75... OK (13.7 MB, 898s)
  [6/14] min 75–90... OK (13.6 MB, 894s)
  [7/14] min 90–105... OK (13.7 MB, 900s)
  [8/14] min 105–120... OK (13.7 MB, 900s)
  [9/14] min 120–135... OK (13.7 MB, 898s)
  [10/14] min 135–150... OK (13.7 MB, 898s)
  [11/14] min 150–165... OK (13.7 MB, 900s)
  [12/14] min 165–180... OK (13.6 MB, 894s)
  [13/14] min 180–195... OK (13.7 MB, 900s)
  [14/14] min 195–210... OK (4.6 MB, 304s)

Audio limpo: 199.6 min (removidos 0.7 min de silêncio)

ETAPA 2: A transcrever com Whisper...
A carregar modelo 'medium'...


100%|█████████████████████████████████████| 1.42G/1.42G [00:20<00:00, 74.6MiB/s]


[00:00.000 --> 00:14.540]  A
[01:14.540 --> 01:21.260]  terceira palestra que será referida para o Sr. Rui Armando da Silva, Secretário-Executivo
[01:21.260 --> 01:24.460]  da Comissão para Implementação de Normas e Segredo de Estado.
[01:24.460 --> 01:33.260]  De seguida, teremos uma mesa redonda onde teremos cinco oradores que terão como lema
[01:33.260 --> 01:39.140]  moderado o operador da Silvia e como tema modernização da gestão de documento e arquivo,
[01:39.140 --> 01:44.060]  experiências e prática para implementação do arquivo digital na EDM.
[01:44.060 --> 01:49.860]  Temos também quinze minutos de debate e, posto isto, teremos a sessão da premiação
[01:49.860 --> 01:54.460]  das unidades orgânicas que se destacaram na gestão do arquivo.
[01:54.460 --> 02:00.660]  E também teremos aqui a premiação dessas unidades e também o segundo passo, que também
[02:00.660 --> 02:03.700]  será outra premiação destacada na gestão documental.
[02:03.700 --> 02:10.500]  Posto isto, no final